In [1]:
!pip install requests beautifulsoup4 nltk pymupdf gradio plotly pandas matplotlib scikit-learn numpy firebase -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 42.6 MB/s eta 0:00:00


In [2]:
import requests
from bs4 import BeautifulSoup
import re, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import WordNetLemmatizer
import nltk
import fitz
from firebase import firebase
nltk.download('wordnet', quiet=True)
print('All imports loaded successfully')

All imports loaded successfully


# Firebase Connection

In [3]:
FIREBASE_URL = "https://bluberrybee1-default-rtdb.firebaseio.com/"

FBconn = firebase.FirebaseApplication(FIREBASE_URL, None)

INDEX_PATH = "/blueberry_index"

print("Connected to Firebase Realtime Database")

Connected to Firebase Realtime Database


## Part 1: Search Engine
### Step 1: Fetch Academic Papers

In [4]:
PAPERS = {
    1: 'https://doi.org/10.3389/fpls.2024.1428769',
    2: 'https://doi.org/10.24266/0738-2898-33.1.33',
    3: 'http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf',
    4: 'https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0283137&type=printable',
    5: 'https://doi.org/10.3389/fpls.2015.00782'
}

def fetch_page(url):
    """Fetch a web page or PDF and return a BeautifulSoup object"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        r = requests.get(url, headers=headers, timeout=10, allow_redirects=True)
        if r.status_code == 200:
            if 'application/pdf' in r.headers.get('Content-Type','') or url.endswith('.pdf'):
                doc = fitz.open(stream=r.content, filetype='pdf')
                return BeautifulSoup(''.join(p.get_text() for p in doc), 'html.parser')
            return BeautifulSoup(r.text, 'html.parser')
        print(f'  Error {r.status_code} for {url}'); return None
    except Exception as e:
        print(f'  Error: {e}'); return None

print('Fetching papers...')
pages = {}
for doc_id, url in PAPERS.items():
    print(f'Paper {doc_id}: {url}')
    pages[doc_id] = fetch_page(url)
    print(f"  {'OK' if pages[doc_id] else 'FAILED'}")

Fetching papers...
Paper 1: https://doi.org/10.3389/fpls.2024.1428769
  OK
Paper 2: https://doi.org/10.24266/0738-2898-33.1.33
  OK
Paper 3: http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf
  OK
Paper 4: https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0283137&type=printable
  OK
Paper 5: https://doi.org/10.3389/fpls.2015.00782
  OK


### Step 2: Build Word Index

In [5]:
def index_words(soup):
    index = {}
    if soup is None: return index
    for w in re.findall(r'\w+', soup.get_text()):
        w = w.lower(); index[w] = index.get(w,0) + 1
    return index

print('Building indices...')
paper_indices = {}
for doc_id, soup in pages.items():
    paper_indices[doc_id] = index_words(soup)
    print(f'Paper {doc_id}: {len(paper_indices[doc_id])} unique words')

Building indices...
Paper 1: 4185 unique words
Paper 2: 3393 unique words
Paper 3: 3592 unique words
Paper 4: 1618 unique words
Paper 5: 3468 unique words


### Step 3: Stop Words & Lemmatization

In [6]:
STOP_WORDS = {
    'a','an','the','and','or','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had',
    'do','does','did','will','would','could','should','may','might',
    'this','that','these','those','by','from','as','it','its','not',
    'but','we','our','their','they','he','she','his','her'
}

def remove_stop_words(index):
    return {w:c for w,c in index.items() if w not in STOP_WORDS}

def apply_lemmatization(index):
    """Reduce words to their base dictionary form using WordNetLemmatizer"""
    lem = WordNetLemmatizer()
    out = {}
    for word, count in index.items():
        base = lem.lemmatize(word)
        out[base] = out.get(base,0) + count
    return out

print('Processing (stop words + lemmatization)...')
for doc_id in paper_indices:
    paper_indices[doc_id] = remove_stop_words(paper_indices[doc_id])
    paper_indices[doc_id] = apply_lemmatization(paper_indices[doc_id])
    print(f'Paper {doc_id}: {len(paper_indices[doc_id])} words after processing')

Processing (stop words + lemmatization)...
Paper 1: 3943 words after processing
Paper 2: 3147 words after processing
Paper 3: 3349 words after processing
Paper 4: 1517 words after processing
Paper 5: 3264 words after processing


### Step 4: Build Main Index (20 Significant Terms)

# Step 3.1: Firebase Functions for Saving and Loading the Index

In [7]:
def save_index_to_firebase(index):
    FBconn.delete(INDEX_PATH, None)
    for term, data in index.items():
        data_to_upload = {
            "term": data["term"],
            "DocIDs": data["DocIDs"]
        }
        FBconn.put(INDEX_PATH, term, data_to_upload)
    print("Index saved to Firebase successfully.")


def load_index_from_firebase():
    index = FBconn.get(INDEX_PATH, None)
    if index:
        print("Index loaded from Firebase.")
        return index
    print("No index found in Firebase.")
    return None

In [8]:
TERMS = [
    'blueberry','vaccinium','anthocyanin','phenolic','cultivar',
    'ripening','maturity','harvest','quality','disease',
    'botrytis','anthracnose','colletotrichum','rot','resistance',
    'breeding','irrigation','fertilizer','drought','soil'
]

lemmatizer = WordNetLemmatizer()

def build_main_index(terms, paper_indices):
    """Build inverted index: term -> {term, DocIDs}"""
    main_index = {}
    for term in terms:
        lemma = lemmatizer.lemmatize(term.lower())
        doc_ids = [did for did,idx in paper_indices.items() if lemma in idx]
        main_index[term] = {'term': term, 'DocIDs': doc_ids}
    return main_index

main_index = load_index_from_firebase()

if main_index is None:
    print('Building index from papers...')
    main_index = build_main_index(TERMS, paper_indices)
    save_index_to_firebase(main_index)
else:
    print('Using existing index from Firebase. No need to rebuild.')
print(f"{'term':<20} {'DocIDs'}")
print('-'*45)
for term, data in main_index.items():
    print(f"{term:<20} {data['DocIDs']}")

Index loaded from Firebase.
Using existing index from Firebase. No need to rebuild.
term                 DocIDs
---------------------------------------------
anthocyanin          [2, 3, 4, 5]
anthracnose          [1, 3]
blueberry            [1, 2, 3, 4, 5]
botrytis             [1, 3]
breeding             [1, 2, 3, 5]
colletotrichum       [1, 3]
cultivar             [1, 2, 3, 5]
disease              [1, 2, 3, 5]
drought              [1, 2, 3, 5]
fertilizer           [2, 3, 4]
harvest              [1, 2, 3, 5]
irrigation           [1, 2, 3, 5]
maturity             [1, 2, 4, 5]
phenolic             [1, 3, 4, 5]
quality              [1, 2, 3, 4, 5]
resistance           [1, 2, 3, 5]
ripening             [1, 2, 3, 4, 5]
rot                  [1, 2, 3]
soil                 [1, 2, 3, 4, 5]
vaccinium            [1, 2, 3, 4, 5]


### Step 5: Search Function

In [9]:
def search(query, paper_indices):
    """Search papers by query and return relevance-ranked results"""
    lem = WordNetLemmatizer()
    q_words = [lem.lemmatize(w) for w in re.findall(r'\w+', query.lower())]
    scores = {}
    for did, idx in paper_indices.items():
        found = {w: idx[w] for w in q_words if w in idx}
        if found:
            rank = 1 - 1/(sum(found.values())+1)
            scores[did] = {'results': found, 'rank': rank}
    return sorted(scores.items(), key=lambda x: x[1]['rank'], reverse=True)

print("Test: 'blueberry disease'")
for did, data in search('blueberry disease', paper_indices):
    print(f'  Paper {did} | Score: {data["rank"]:.4f} | {data["results"]}')

Test: 'blueberry disease'
  Paper 3 | Score: 0.9972 | {'blueberry': 284, 'disease': 70}
  Paper 2 | Score: 0.9960 | {'blueberry': 211, 'disease': 39}
  Paper 1 | Score: 0.9959 | {'blueberry': 179, 'disease': 65}
  Paper 5 | Score: 0.9942 | {'blueberry': 168, 'disease': 3}
  Paper 4 | Score: 0.9905 | {'blueberry': 104}


## Part 1.4: RAG System
Retrieval-Augmented Generation for enriched search results

In [10]:
class SimpleVectorStore:
    """In-memory vector store — no external DB needed"""
    def __init__(self):
        self.documents,self.embeddings,self.metadatas,self.ids=[],[],[],[]

    def add(self,embeddings,documents,metadatas,ids):
        self.embeddings.extend(embeddings);self.documents.extend(documents)
        self.metadatas.extend(metadatas);self.ids.extend(ids)

    def query(self,q_emb,n_results=3):
        if not self.embeddings:
            return {'ids':[[]],'documents':[[]],'metadatas':[[]],'distances':[[]]}
        sims=cosine_similarity(q_emb,self.embeddings)[0]
        top=np.argsort(sims)[::-1][:n_results]
        return {'ids':[[self.ids[i] for i in top]],'documents':[[self.documents[i] for i in top]],
                'metadatas':[[self.metadatas[i] for i in top]],'distances':[[1-sims[i] for i in top]]}

    def count(self): return len(self.documents)


class BlueberryRAG:
    """RAG system using TF-IDF embeddings + in-memory vector store"""
    def __init__(self):
        self.tfidf=TfidfVectorizer(max_features=1000,stop_words='english')
        self.fitted=False; self.collection=SimpleVectorStore()
        self.lemmatizer=WordNetLemmatizer(); self.papers=[]

    def _prep(self,text):
        if not text: return ''
        return re.sub(r'[^\w\s\-\.\(\)]',' ',re.sub(r'\s+',' ',text)).strip().lower()

    def _embed(self,texts):
        if not self.fitted: self.tfidf.fit(texts); self.fitted=True
        return self.tfidf.transform(texts).toarray()

    def load_papers(self,papers):
        docs,metas,ids=[],[],[]
        for p in papers:
            if not p.get('abstract','').strip(): continue
            docs.append(self._prep(f"{p.get('title','')} {p.get('abstract','')}"))
            metas.append({k:str(p.get(k,'')) for k in ('doc_id','title','authors','journal','year','doi')})
            ids.append(f"doc_{p['doc_id']}")
        if docs:
            self.collection.add(self._embed(docs),docs,metas,ids)
            self.papers=papers; print(f'Loaded {len(docs)} papers into RAG')

    def query(self,question,n_results=3):
        if self.collection.count()==0: return {'response':'No papers loaded.','papers_found':0}
        q_emb=self._embed([self._prep(question)])
        res=self.collection.query(q_emb,n_results)
        docs,metas,dists=res['documents'][0],res['metadatas'][0],res['distances'][0]
        ctx='\n\n'.join(
            f"[DocID {m['doc_id']}] \"{m['title']}\" ({m['authors']},{m['year']})"
            f" — Relevance:{round((1-d)*100,1)}%\nExcerpt:{doc[:250]}..."
            for doc,m,d in zip(docs,metas,dists)
        )
        kw=[self.lemmatizer.lemmatize(w) for w in re.findall(r'\w+',question.lower()) if len(w)>3]
        resp=(f"Query: \"{question}\"\n{'='*60}\n\n"
              f"Found {len(docs)} paper(s):\n\n{ctx}\n\n"
              f"{'='*60}\nKey terms: {', '.join(kw)}\n"
              f"Top match: \"{metas[0]['title']}\" ({round((1-dists[0])*100,1)}%)")
        return {'response':resp,'papers_found':len(docs),'raw':res}


BLUEBERRY_PAPERS=[
  {'doc_id':1,'title':'Managing fruit rot diseases of Vaccinium corymbosum','authors':'Neugebauer et al.',
   'journal':'Frontiers in Plant Science','year':2024,'doi':'https://doi.org/10.3389/fpls.2024.1428769',
   'abstract':'Blueberry pre- and post-harvest diseases including anthracnose (Colletotrichum spp.) and botrytis fruit rot (Botrytis spp.) significantly impact fruit quality. Management innovations include molecular detection for fungicide resistance, postharvest imaging, breeding resistant cultivars, and biopesticides.'},
  {'doc_id':2,'title':'Blueberry Culture and Pest, Disease, and Abiotic Disorder Management','authors':'Fulcher et al.',
   'journal':'HortTechnology','year':2015,'doi':'https://doi.org/10.24266/0738-2898-33.1.33',
   'abstract':'Vaccinium blueberries ranked second most important berry crop in the U.S. This review covers culture, insect, mite, disease control via integrated pest management (IPM) for highbush, lowbush, rabbiteye, and southern highbush species. Spotted wing drosophila is a serious emerging threat.'},
  {'doc_id':3,'title':'Highbush Blueberry: Cultivation, Protection, Breeding and Biotechnology','authors':'Prodorutti et al.',
   'journal':'European Journal of Plant Science and Biotechnology','year':2007,
   'doi':'http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf',
   'abstract':'Highbush blueberry is one of the most commercially significant berry crops. Review covers agronomical techniques including mulching, irrigation, mycorrhizae, fertilization, disease management, pest management, pollinators, conventional breeding and molecular techniques.'},
  {'doc_id':4,'title':'Effects of NPK fertilization on yield and berry quality of blueberry','authors':'Zhang et al.',
   'journal':'PLOS ONE','year':2023,'doi':'https://doi.org/10.1371/journal.pone.0283137',
   'abstract':'NPK fertilizer ratios were studied for effects on blueberry yield and quality. K fertilizer was the most important factor. Fertilization significantly improved yield, anthocyanins, total phenols, soluble solids content compared to the control group.'},
  {'doc_id':5,'title':'Breeding blueberries for a changing global environment','authors':'Lobos & Hancock',
   'journal':'Frontiers in Plant Science','year':2015,'doi':'https://doi.org/10.3389/fpls.2015.00782',
   'abstract':'Blueberries are recognized worldwide as a top health food. Highbush growing areas now extend into hotter and drier environments. Genetic variability in the blueberry gene pool supports drought resistance breeding. Marker assisted breeding (MAB) and phenomics could increase selection efficiency.'}
]

rag_system=BlueberryRAG()
rag_system.load_papers(BLUEBERRY_PAPERS)
print('RAG system ready!')

Loaded 5 papers into RAG
RAG system ready!


## Part 2: Screens
### Screen 2 Backend: IoT Sensor Functions

In [11]:
BASE_URL = 'https://server-cloud-v645.onrender.com/'

def fetch_sensor_data(feed, limit):
    try:
        r=requests.get(f'{BASE_URL}/history',params={'feed':feed,'limit':limit},timeout=60)
        data=r.json()
        if 'data' not in data: return None,f'Error:{data}'
        df=pd.DataFrame(data['data'])
        df['created_at']=pd.to_datetime(df['created_at'])
        df['value']=pd.to_numeric(df['value'],errors='coerce')
        return df,'OK'
    except Exception as e: return None,str(e)

def evaluate_sensor_status(feed,value):
    if feed=='temperature':
        u='°C'
        if 18<=value<=26:          return '🟢','Healthy','Temperature is suitable for blueberry growth.',u
        elif 15<=value<18 or value<=30: return '🟡','Warning','Temperature slightly outside ideal range.',u
        else:                      return '🔴','Critical','Temperature may stress the blueberry plant.',u
    if feed=='humidity':
        u='%'
        if 50<=value<=70:          return '🟢','Healthy','Air humidity is suitable for blueberries.',u
        elif 40<=value<50 or value<=80: return '🟡','Warning','Humidity is not ideal. Keep monitoring.',u
        else:                      return '🔴','Critical','Humidity may increase disease risk.',u
    if feed=='soil':
        u='%'
        if 50<=value<=70:          return '🟢','Healthy','Soil moisture is balanced for blueberries.',u
        elif 35<=value<50 or value<=85: return '🟡','Warning','Soil moisture not ideal. Check drainage.',u
        else:                      return '🔴','Critical','Soil moisture risky. Immediate check needed.',u
    return '⚪','Unknown','No data available.',''

def get_sensor_history(feed,limit):
    df,msg=fetch_sensor_data(feed,int(limit))
    if df is None: return f'<p style="color:red">Error: {msg}</p>',None,None
    v=df['value'].iloc[0]; t=df['created_at'].iloc[0].strftime('%Y-%m-%d %H:%M')
    icon,status,rec,unit=evaluate_sensor_status(feed,v)
    cls={'Healthy':'healthy-card','Warning':'warning-card'}.get(status,'critical-card')
    html=(f'<div class="status-card {cls}">'
          f'<div class="status-icon">{icon}</div><div class="status-title">{status}</div>'
          f'<div class="status-value">{v} {unit}</div><div class="status-subtitle">Current {feed.capitalize()} Reading</div>'
          f'<div class="status-row"><span>Last Update</span><strong>{t}</strong></div>'
          f'<div class="status-row"><span>Samples</span><strong>{len(df)}</strong></div>'
          f'<div class="recommendation-box">{rec}</div></div>')
    fig,ax=plt.subplots(figsize=(9,4.5))
    fig.patch.set_facecolor('#faf7ff');ax.set_facecolor('#ffffff')
    ax.plot(df['created_at'],df['value'],marker='o',linewidth=2.5,color='#5b3cc4',
            markerfacecolor='#2f195f',markeredgecolor='white',markeredgewidth=1.5)
    ax.fill_between(df['created_at'],df['value'],alpha=0.15,color='#5b3cc4')
    ax.set_title(f'{feed.capitalize()} Trend Over Time',fontsize=14,fontweight='bold')
    ax.set_xlabel('Time');ax.set_ylabel(f'Value ({unit})')
    ax.grid(True,alpha=0.3);ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False)
    plt.xticks(rotation=35);plt.tight_layout()
    tbl=df[['created_at','value']].copy()
    tbl['created_at']=tbl['created_at'].dt.strftime('%d/%m/%Y %H:%M')
    tbl.columns=['Date & Time',f'Value ({unit})']
    return html,fig,tbl

print('Sensor functions ready')

Sensor functions ready


### Screen 4 Backend: Dashboard Functions

In [12]:
def fetch_dashboard_data(limit=30):
    out={}
    for f in ['temperature','humidity','soil']:
        df,_=fetch_sensor_data(f,limit)
        if df is not None and len(df)>0: out[f]=df
    return out

def calculate_health_score(data):
    scores=[]
    for feed,df in data.items():
        _,status,_,_=evaluate_sensor_status(feed,df['value'].iloc[0])
        scores.append(100 if status=='Healthy' else 65 if status=='Warning' else 30)
    return round(sum(scores)/len(scores)) if scores else 0

def get_health_label(score):
    if score>=80: return '🟢 Excellent','Your blueberry plant looks healthy overall.'
    if score>=60: return '🟡 Needs Monitoring','Some conditions are not ideal. Keep monitoring.'
    return '🔴 Needs Attention','Several conditions may stress the plant.'

def get_daily_mission(score):
    if score>=80: return '🌟 Keep It Healthy','Maintain current care routine.','+20 points'
    if score>=60: return '🔍 Monitor Closely','Review sensor with lowest condition.','+35 points'
    return '🚨 Recovery Mission','Take action on critical sensors.','+50 points'

def create_dashboard_cards(data):
    score=calculate_health_score(data)
    t=data.get('temperature',pd.DataFrame([{'value':0}]))['value'].iloc[0]
    h=data.get('humidity',   pd.DataFrame([{'value':0}]))['value'].iloc[0]
    s=data.get('soil',       pd.DataFrame([{'value':0}]))['value'].iloc[0]
    return (f'<div class="dashboard-grid">'
            f'<div class="dashboard-card"><div class="dashboard-icon">🌡</div><div class="dashboard-title">Temperature</div><div class="dashboard-value">{t} °C</div><div class="dashboard-subtitle">Latest reading</div></div>'
            f'<div class="dashboard-card"><div class="dashboard-icon">💧</div><div class="dashboard-title">Humidity</div><div class="dashboard-value">{h} %</div><div class="dashboard-subtitle">Latest reading</div></div>'
            f'<div class="dashboard-card"><div class="dashboard-icon">🌱</div><div class="dashboard-title">Soil Moisture</div><div class="dashboard-value">{s} %</div><div class="dashboard-subtitle">Latest reading</div></div>'
            f'<div class="dashboard-card health-score-card"><div class="dashboard-icon">🫐</div><div class="dashboard-title">Plant Health Score</div><div class="dashboard-value">{score}%</div><div class="dashboard-subtitle">Overall condition</div></div>'
            '</div>')

def create_dashboard_plot(data):
    fig,ax=plt.subplots(figsize=(11,4.5))
    fig.patch.set_facecolor('#faf7ff');ax.set_facecolor('#ffffff')
    for feed,df in data.items():
        ax.plot(df['created_at'],df['value'],marker='o',linewidth=2,label=feed.capitalize())
    ax.set_title('Sensor Trends Comparison',fontsize=15,fontweight='bold')
    ax.set_xlabel('Time');ax.set_ylabel('Value');ax.grid(True,alpha=0.25)
    ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False)
    ax.legend();plt.xticks(rotation=35);plt.tight_layout()
    return fig

def load_dashboard(sel):
    lim={'today':30,'yesterday':60,'week':100}.get(sel,30)
    data=fetch_dashboard_data(lim)
    if not data: return '<p>Could not load sensor data. Please check the server.</p>',None,''
    cards=create_dashboard_cards(data)
    plot=create_dashboard_plot(data)
    score=calculate_health_score(data)
    label,msg=get_health_label(score)
    mt,mx,mp=get_daily_mission(score)
    rec=(f'<div class="dashboard-recommendation"><h3>🫐 Blueberry Care Summary</h3>'
         f'<p>Current health score: <strong>{score}%</strong></p>'
         f'<p><strong>{label}</strong> — {msg}</p>'
         f'<div class="gamification-box"><h4>🏆 {mt}</h4><p>{mx}</p><strong>{mp}</strong></div></div>')
    return cards,plot,rec

print('Dashboard functions ready')

Dashboard functions ready


### Screen 1 Backend: Plant Image Upload

In [13]:
uploaded_images = []

def upload_plant_image(image, plant_name, health_status, notes):
    """Save an uploaded plant image with metadata"""
    if image is None: return 'Please upload an image.', build_image_gallery()
    if not plant_name.strip(): return 'Please enter a plant name.', build_image_gallery()
    uploaded_images.append({
        'image': image, 'plant_name': plant_name.strip(),
        'health_status': health_status, 'notes': notes.strip(),
        'uploaded_at': datetime.now().strftime('%Y-%m-%d %H:%M')
    })
    return f"Image saved for '{plant_name}' ({health_status}).", build_image_gallery()

def build_image_gallery():
    return [(e['image'], f"{e['plant_name']} — {e['health_status']} | {e['uploaded_at']}")
            for e in uploaded_images]

def get_images_table():
    if not uploaded_images:
        return pd.DataFrame(columns=['Plant','Status','Notes','Uploaded At'])
    return pd.DataFrame([{'Plant':e['plant_name'],'Status':e['health_status'],
                           'Notes':e['notes'][:60]+('...' if len(e['notes'])>60 else ''),
                           'Uploaded At':e['uploaded_at']} for e in uploaded_images])

print('Image upload functions ready')

Image upload functions ready


### Screen 3 Backend: Search Query with RAG

In [14]:
PAPER_URLS = {
    '1':'https://doi.org/10.3389/fpls.2024.1428769',
    '2':'https://doi.org/10.24266/0738-2898-33.1.33',
    '3':'http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf',
    '4':'https://doi.org/10.1371/journal.pone.0283137',
    '5':'https://doi.org/10.3389/fpls.2015.00782'
}

def run_search(query, n_results):
    """Run RAG search and return enriched HTML results"""
    if not query.strip(): return '<p>Please enter a search query.</p>'
    result=rag_system.query(query, n_results=int(n_results))
    raw=result.get('raw',{})
    if not raw or not raw.get('documents',[[]])[0]:
        return '<p>No results found.</p>'
    metas,dists,docs=raw['metadatas'][0],raw['distances'][0],raw['documents'][0]
    cards=[]
    for meta,dist,doc in zip(metas,dists,docs):
        sim=round((1-dist)*100,1)
        bar='█'*int(sim/5)+'░'*(20-int(sim/5))
        url=PAPER_URLS.get(str(meta['doc_id']),'#')
        ex=doc[:300].replace('<','&lt;').replace('>','&gt;')
        cards.append(
            f'<div class="search-result-card">'
            f'<div class="search-result-header"><span class="doc-id">DocID {meta["doc_id"]}</span><span class="relevance-score">{sim}% match</span></div>'
            f'<div class="search-result-title">{meta["title"]}</div>'
            f'<div class="search-result-meta">{meta["authors"]} · {meta["journal"]} · {meta["year"]}</div>'
            f'<div class="relevance-bar">{bar}</div>'
            f'<div class="search-excerpt">"{ex}..."</div>'
            f'<a class="doi-link" href="{url}" target="_blank">🔗 Open Paper (DOI)</a>'
            '</div>'
        )
    kw=[lemmatizer.lemmatize(w) for w in re.findall(r'\w+',query.lower()) if len(w)>3]
    hdr=(f'<div class="search-header"><h2>🔍 Results for: <em>"{query}"</em></h2>'
         f'<p>Found <strong>{len(metas)}</strong> paper(s) · Key terms: {", ".join(kw)}</p></div>')
    return hdr+''.join(cards)

print('Search/RAG function ready')

Search/RAG function ready


## Part 3: Extra Feature: Before & After Gallery

In [15]:
cases_data=[]

def add_case(plant_type,disease,before_img,after_img,treatment,result):
    """Add a before/after treatment case to the gallery"""
    if not plant_type or not disease or before_img is None or after_img is None or not treatment:
        return 'Please fill all required fields.',pd.DataFrame(cases_data)
    cases_data.append({
        'Date':datetime.now().strftime('%Y-%m-%d %H:%M'),'Plant Type':plant_type,
        'Disease / Problem':disease,'Before Image':before_img,
        'After Image':after_img,'Treatment':treatment,'Result':result
    })
    df=pd.DataFrame(cases_data).drop(columns=['Before Image','After Image'])
    return 'Case added successfully!',df

def search_cases(plant_type):
    """Search cases by plant type and return results with image gallery"""
    if not cases_data: return 'No cases added yet.',pd.DataFrame(),[]
    df=pd.DataFrame(cases_data)
    fil=df[df['Plant Type'].str.lower().str.contains(plant_type.lower())] if plant_type else df
    if fil.empty: return 'No similar cases found.',pd.DataFrame(),[]
    gallery=[]
    for _,row in fil.iterrows():
        gallery.append((row['Before Image'],'Before Treatment'))
        gallery.append((row['After Image'],'After Treatment'))
    return 'Cases found:',fil.drop(columns=['Before Image','After Image']),gallery

print('Before & After functions ready')

Before & After functions ready


### CSS Styles

In [16]:
custom_css = '''

/* Base layout */
.gradio-container, main {
    background: #F6F4FB !important;
    color: #1F1B2E !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
    padding: 0 !important;
}

/* Hide tab bar (navigation handled by custom buttons) */
[role='tablist'] {
    height: 0 !important;
    min-height: 0 !important;
    overflow: hidden !important;
    opacity: 0 !important;
    margin: 0 !important;
    padding: 0 !important;
}

/* Home screen: hero image */
.home-hero {
    width: 100%;
    height: 100vh;
    min-height: 650px;
    overflow: hidden;
    position: relative;
    margin: 0;
}
.home-bg {
    width: 100%;
    height: 100%;
    object-fit: cover;
    object-position: center;
    display: block;
    filter: saturate(1.1) contrast(1.05);
}
.home-overlay {
    position: absolute;
    inset: 0;
    background: radial-gradient(circle at 30% 30%, rgba(91,60,196,.20), transparent 35%),
                linear-gradient(rgba(8,10,30,.45), rgba(8,10,30,.70));
    display: flex;
    flex-direction: column;
    justify-content: center;
    align-items: center;
    padding: clamp(24px, 5vw, 50px);
    text-align: center;
}
.home-logo { font-size: clamp(48px,6vw,76px); margin-bottom: 8px; }
.home-overlay h1 {
    color: white !important;
    font-size: clamp(38px, 6vw, 72px);
    font-weight: 900;
    margin: 0;
    text-shadow: 0 6px 18px rgba(0,0,0,.45);
}
.home-overlay p {
    color: white !important;
    font-size: clamp(16px, 2vw, 24px);
    max-width: 760px;
    line-height: 1.55;
    margin-top: 22px;
    text-shadow: 0 3px 12px rgba(0,0,0,.55);
}

/* Home buttons */
.home-buttons {
    display: flex;
    gap: 24px;
    margin-top: 40px;
    flex-wrap: wrap;
    justify-content: center;
}
.home-action-btn {
    min-width: min(260px, 90vw);
    padding: 16px 26px;
    border-radius: 22px;
    font-size: clamp(14px, 2vw, 19px);
    font-weight: 850;
    color: white !important;
    cursor: pointer;
    backdrop-filter: blur(14px);
    transition: all .25s ease;
}
.purple-btn { background: rgba(91,60,196,.30) !important; border: 2px solid rgba(167,139,250,.85) !important; }
.blue-btn   { background: rgba(37,99,235,.28)  !important; border: 2px solid rgba(96,165,250,.9)   !important; }
.green-btn  { background: rgba(5,150,105,.30)  !important; border: 2px solid rgba(52,211,153,.85)  !important; }
.orange-btn { background: rgba(217,119,6,.30)  !important; border: 2px solid rgba(251,191,36,.85)  !important; }
.teal-btn   { background: rgba(13,148,136,.30) !important; border: 2px solid rgba(45,212,191,.85)  !important; }
.home-action-btn:hover {
    transform: translateY(-4px);
    background: rgba(255,255,255,.20) !important;
}
.home-footer {
    position: absolute;
    bottom: 0; left: 0; right: 0;
    display: flex;
    gap: 18px;
    justify-content: center;
    align-items: center;
    color: white !important;
    font-size: clamp(12px, 1.4vw, 16px);
    padding: 18px;
    background: rgba(8,10,30,.26);
    backdrop-filter: blur(10px);
    border-top: 1px solid rgba(255,255,255,.15);
    flex-wrap: wrap;
}
.home-footer div, .home-footer span { color: white !important; }

/* Screen wrapper */
.screen-wrapper { padding: 28px; }
#title-box {
    background: linear-gradient(135deg, #FFFFFF, #EFE9FF);
    padding: 34px 30px;
    border-radius: 28px;
    margin-bottom: 24px;
    box-shadow: 0 14px 35px rgba(76,42,133,.12);
    border: 1px solid #E6DCF8;
}
#title-box h1 { font-size: 42px; margin: 0; font-weight: 850; color: #2F195F; }
#title-box p  { font-size: 18px; margin-top: 10px; color: #6B5A86; }
.helper-box {
    background: #FFFFFF;
    color: #4C3A67;
    border-radius: 20px;
    padding: 16px 20px;
    margin: 12px 0 20px;
    font-size: 15px;
    font-weight: 500;
    border: 1px solid #E6DCF8;
    box-shadow: 0 8px 20px rgba(47,25,95,.06);
}

/* Cards and panels */
.block, .form, .panel {
    background: #FFFFFF !important;
    border-radius: 22px !important;
    border: 1px solid #E8E2F2 !important;
    box-shadow: 0 10px 28px rgba(47,25,95,.08) !important;
    color: #1F1B2E !important;
    overflow: hidden !important;
}
label, span, p { color: #1F1B2E !important; }

/* Buttons */
button {
    background: #FFFFFF !important;
    color: #3B2474 !important;
    border: 1px solid #DDD3F3 !important;
    border-radius: 18px !important;
    font-weight: 700 !important;
}
button.primary, #load-button {
    background: linear-gradient(135deg, #5B3CC4, #7C5CE6) !important;
    color: white !important;
    border: none !important;
    border-radius: 18px !important;
    font-weight: 800 !important;
    box-shadow: 0 10px 22px rgba(91,60,196,.25) !important;
}
.back-home-btn {
    background: #FFFFFF !important;
    color: #2F195F !important;
    border: 1px solid #DDD3F3 !important;
    border-radius: 16px !important;
    padding: 12px 18px !important;
    font-weight: 800 !important;
    margin-bottom: 18px !important;
    cursor: pointer !important;
}

/* Inputs */
input, textarea, select {
    background: #FFFFFF !important;
    color: #1F1B2E !important;
    border-color: #DDD3F3 !important;
}

/* Radio buttons: make selection visible */
input[type='radio'] { accent-color: #5B3CC4 !important; }
input[type='range']  { accent-color: #5B3CC4 !important; }
fieldset { border: none !important; gap: 12px !important; }
fieldset label {
    background: #FFFFFF !important;
    border: 1px solid #E6DCF8 !important;
    border-radius: 18px !important;
    padding: 18px !important;
    min-height: 70px !important;
    cursor: pointer !important;
}
fieldset label:has(input[type='radio']:checked) {
    border-color: #5B3CC4 !important;
    background: #F3EEFF !important;
}
fieldset input[type='radio'] {
    display: inline-block !important;
    opacity: 1 !important;
    visibility: visible !important;
    width: 16px !important;
    height: 16px !important;
    margin-right: 10px !important;
    accent-color: #5B3CC4 !important;
}

/* Sensor status cards */
.status-card {
    background: #FFFFFF;
    border-radius: 26px;
    padding: 26px;
    min-height: 280px;
    box-shadow: 0 14px 32px rgba(47,25,95,.10);
    border: 1px solid #E8E2F2;
}
.status-icon  { font-size: 38px; margin-bottom: 10px; }
.status-title { font-size: 22px; font-weight: 850; margin-bottom: 8px; }
.status-value { font-size: 48px; font-weight: 900; color: #2F195F; margin-bottom: 8px; }
.status-subtitle { color: #7A6B91; font-size: 14px; margin-bottom: 18px; }
.status-row {
    display: flex;
    justify-content: space-between;
    padding: 11px 0;
    border-top: 1px solid #EFEAF7;
}
.recommendation-box {
    margin-top: 18px;
    padding: 15px;
    border-radius: 18px;
    background: #FFFFFF;
    color: #2F195F;
    font-weight: 700;
}
.healthy-card  { background: #F1FBF4; }
.warning-card  { background: #FFF8E6; }
.critical-card { background: #FFF0F0; }

/* Dashboard */
.dashboard-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 18px;
    margin-bottom: 20px;
}
.dashboard-card {
    background: #FFFFFF;
    border-radius: 24px;
    padding: 22px;
    text-align: center;
    box-shadow: 0 10px 24px rgba(47,25,95,.08);
    border: 1px solid #E8E2F2;
}
.dashboard-icon     { font-size: 32px; margin-bottom: 10px; }
.dashboard-title    { font-size: 16px; font-weight: 700; color: #6B5A86; }
.dashboard-value    { font-size: 34px; font-weight: 900; color: #2F195F; margin-top: 8px; }
.dashboard-subtitle { font-size: 13px; color: #8B7CA5; margin-top: 8px; }
.health-score-card  { background: linear-gradient(135deg, #5B3CC4, #7C5CE6); }
.health-score-card .dashboard-title,
.health-score-card .dashboard-value,
.health-score-card .dashboard-subtitle { color: white !important; }
.dashboard-recommendation {
    background: #FFFFFF;
    border-radius: 24px;
    padding: 22px;
    margin-top: 18px;
    border: 1px solid #E8E2F2;
}
.gamification-box {
    margin-top: 18px;
    padding: 16px;
    border-radius: 18px;
    background: #F3EEFF;
    border: 1px solid #E6DCF8;
}
th { background: #F3EEFF !important; color: #2F195F !important; font-weight: 800 !important; }

/* Search results */
.search-result-card {
    background: #FFFFFF;
    border: 1px solid #E8E2F2;
    border-radius: 20px;
    padding: 22px;
    margin-bottom: 16px;
    box-shadow: 0 8px 20px rgba(47,25,95,.07);
}
.search-result-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 10px;
}
.doc-id         { background: #F3EEFF; color: #5B3CC4; padding: 4px 12px; border-radius: 20px; font-weight: 800; font-size: 13px; }
.relevance-score { color: #059669; font-weight: 800; font-size: 14px; }
.search-result-title { font-size: 18px; font-weight: 850; color: #2F195F; margin-bottom: 6px; }
.search-result-meta  { font-size: 13px; color: #7A6B91; margin-bottom: 12px; }
.relevance-bar  { font-family: monospace; font-size: 13px; color: #5B3CC4; letter-spacing: 1px; margin-bottom: 10px; }
.search-excerpt {
    background: #FAFAFA;
    border-left: 3px solid #5B3CC4;
    padding: 10px 14px;
    border-radius: 0 10px 10px 0;
    font-size: 14px;
    color: #444;
    margin-bottom: 12px;
}
.doi-link     { color: #5B3CC4; font-weight: 700; text-decoration: none; font-size: 14px; }
.search-header { margin-bottom: 20px; padding: 20px; background: #F3EEFF; border-radius: 16px; }
.search-header h2 { margin: 0 0 8px; color: #2F195F !important; font-size: 22px; }
.search-header p  { margin: 0; color: #5B3CC4 !important; }

/* Responsive */
@media (max-width: 900px) { .dashboard-grid { grid-template-columns: repeat(2, 1fr); } }
@media (max-width: 700px) {
    .home-buttons { flex-direction: column; width: 100%; align-items: center; }
    .dashboard-grid { grid-template-columns: 1fr; }
}

'''


## Main Gradio Application: All 4 Screens

In [17]:
with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as app:
    with gr.Tabs():

        # HOME
        with gr.Tab('Home',id='home'):
            gr.HTML("""
<div class='home-hero'>
  <img class='home-bg' src='https://i.pinimg.com/736x/7f/73/21/7f73213f3f1869402396fa8c8d4aab10.jpg'>
  <div class='home-overlay'>
    <div class='home-logo'>🫐</div>
    <h1>Blueberry Smart Care</h1>
    <p>Monitor your blueberry plant health, analyze sensor trends and receive smart recommendations.</p>
    <div class='home-buttons'>
      <button class='home-action-btn green-btn'  onclick="document.querySelectorAll('button[role=tab]')[1].click()">📷 Plant Images</button>
      <button class='home-action-btn purple-btn' onclick="document.querySelectorAll('button[role=tab]')[2].click()">📡 Sensors</button>
      <button class='home-action-btn orange-btn' onclick="document.querySelectorAll('button[role=tab]')[3].click()">🔍 Search Papers</button>
      <button class='home-action-btn blue-btn'   onclick="document.querySelectorAll('button[role=tab]')[4].click()">📊 Dashboard</button>
      <button class='home-action-btn teal-btn'   onclick="document.querySelectorAll('button[role=tab]')[5].click()">🌿 Gallery</button>
    </div>
    <div class='home-footer'>
      <div>📡 Real-Time IoT Monitoring</div><span>·</span>
      <div>🧠 Smart RAG Search</div><span>·</span>
      <div>🌿 Better Plant Health</div>
    </div>
  </div>
</div>""")

        # SCREEN 1: Image Upload
        with gr.Tab('Images',id='images'):
            gr.HTML("""<div class='screen-wrapper'>
<button class='back-home-btn' onclick="document.querySelectorAll('button[role=tab]')[0].click()">← Back Home</button>
<div id='title-box'><h1>📷 Plant Image Upload</h1><p>Upload and track photos of your blueberry plant over time</p></div>
<div class='helper-box'>💡 Upload a photo, choose health status, and add notes. All images are saved in the gallery below.</div>
</div>""")
            with gr.Row():
                with gr.Column(scale=1):
                    img_input=gr.Image(label='Plant Photo',type='filepath')
                    plant_name_in=gr.Textbox(label='Plant Name / ID',placeholder='e.g. Blueberry Bush #1')
                    health_in=gr.Radio(choices=['🟢 Healthy','🟡 Needs Attention','🔴 Critical'],
                                       value='🟢 Healthy',label='Health Status')
                    notes_in=gr.Textbox(label='Observations / Notes',lines=3,placeholder='Describe what you see...')
                    upload_btn=gr.Button('💾 Save Image',variant='primary')
                    upload_msg=gr.Textbox(label='Status',interactive=False)
                with gr.Column(scale=2):
                    img_gallery=gr.Gallery(label='Uploaded Plant Images',columns=3,height=400)
            images_table=gr.Dataframe(label='Image Log')
            upload_btn.click(fn=upload_plant_image,inputs=[img_input,plant_name_in,health_in,notes_in],
                             outputs=[upload_msg,img_gallery])
            upload_btn.click(fn=get_images_table,inputs=[],outputs=[images_table])

        # SCREEN 2: Sensors
        with gr.Tab('Sensors',id='sensors'):
            gr.HTML("""<div class='screen-wrapper'>
<button class='back-home-btn' onclick="document.querySelectorAll('button[role=tab]')[0].click()">← Back Home</button>
<div id='title-box'><h1>🫐 Blueberry Plant Monitor</h1><p>Real-Time Cloud Monitoring & Health Analysis</p></div>
<div class='helper-box'>💡 Choose a sensor and how many samples to analyze. See current value, trend graph, and blueberry care recommendation.</div>
</div>""")
            with gr.Row():
                feed_input=gr.Radio(choices=[('💧 Humidity','humidity'),('🌱 Soil Moisture','soil'),('🌡 Temperature','temperature')],
                                    value='humidity',label='Choose Sensor')
                limit_input=gr.Slider(minimum=1,maximum=100,value=30,step=1,label='Number of Samples')
            with gr.Row():
                check_button=gr.Button('📡 Load Sensor Data',variant='primary',elem_id='load-button')
                reset_button=gr.Button('🔄 Reset')
            with gr.Row():
                with gr.Column(scale=1): status_output=gr.HTML(label='Sensor Status')
                with gr.Column(scale=2): plot_output=gr.Plot(label='Trend Graph')
            table_output=gr.Dataframe(label='Recent Measurements')
            check_button.click(fn=get_sensor_history,inputs=[feed_input,limit_input],
                               outputs=[status_output,plot_output,table_output],show_progress=True)
            reset_button.click(fn=lambda:('',None,None),inputs=[],outputs=[status_output,plot_output,table_output])

        # SCREEN 3: Search / RAG
        with gr.Tab('Search',id='search'):
            gr.HTML("""<div class='screen-wrapper'>
<button class='back-home-btn' onclick="document.querySelectorAll('button[role=tab]')[0].click()">← Back Home</button>
<div id='title-box'><h1>🔍 Research Paper Search</h1><p>RAG-powered search across 5 academic blueberry papers</p></div>
<div class='helper-box'>💡 Enter a query about blueberry diseases, cultivation, or breeding. The RAG system retrieves the most relevant papers with enriched results and relevance scores.</div>
</div>""")
            with gr.Row():
                search_query=gr.Textbox(label='Search Query',placeholder='e.g. botrytis disease treatment blueberry',scale=4)
                n_results_in=gr.Slider(minimum=1,maximum=5,value=3,step=1,label='Results',scale=1)
            with gr.Row():
                search_btn=gr.Button('🔍 Search Papers',variant='primary')
                clear_btn=gr.Button('✕ Clear')
            search_output=gr.HTML(label='Search Results')
            search_btn.click(fn=run_search,inputs=[search_query,n_results_in],outputs=[search_output])
            clear_btn.click(fn=lambda:('',''),inputs=[],outputs=[search_query,search_output])

        # SCREEN 4: Dashboard
        with gr.Tab('Dashboard',id='dashboard'):
            gr.HTML("""<div class='screen-wrapper'>
<button class='back-home-btn' onclick="document.querySelectorAll('button[role=tab]')[0].click()">← Back Home</button>
<div id='title-box'><h1>🫐 My Blueberry Dashboard</h1><p>Overall plant condition, sensor summary and care insights</p></div>
</div>""")
            dashboard_range=gr.Radio(choices=[('Today','today'),('Yesterday','yesterday'),('Last 7 Days','week')],
                                     value='today',label='Dashboard View')
            load_dash_btn=gr.Button('📊 Load Dashboard',variant='primary')
            cards_output=gr.HTML()
            dash_plot=gr.Plot(label='Sensor Trends Comparison')
            rec_output=gr.HTML()
            load_dash_btn.click(fn=load_dashboard,inputs=[dashboard_range],
                                outputs=[cards_output,dash_plot,rec_output],show_progress=True)

        # BONUS: Before & After Gallery (Part 3)
        with gr.Tab('Gallery',id='gallery'):
            gr.HTML("""<div class='screen-wrapper'>
<button class='back-home-btn' onclick="document.querySelectorAll('button[role=tab]')[0].click()">← Back Home</button>
<div id='title-box'><h1>🌿 Before & After Gallery</h1><p>Share and explore plant treatment cases</p></div>
<div class='helper-box'>💡 Add a new treatment case with before/after photos, or search past cases by plant type.</div>
</div>""")
            with gr.Tab('Add New Case'):
                plant_type=gr.Textbox(label='Plant Type',placeholder='e.g. Blueberry')
                disease=gr.Textbox(label='Disease / Problem',placeholder='e.g. Yellow leaves')
                before_img=gr.Image(label='Before Treatment',type='filepath')
                after_img=gr.Image(label='After Treatment',type='filepath')
                treatment=gr.Textbox(label='Treatment Used',lines=3)
                result_txt=gr.Textbox(label='Result / Recommendation',lines=3)
                add_btn=gr.Button('Add Case',variant='primary')
                add_status=gr.Textbox(label='Status')
                all_cases=gr.Dataframe(label='All Saved Cases')
                add_btn.click(fn=add_case,inputs=[plant_type,disease,before_img,after_img,treatment,result_txt],
                              outputs=[add_status,all_cases])
            with gr.Tab('Search Cases'):
                search_plant=gr.Textbox(label='Search by Plant Type')
                search_btn2=gr.Button('Search')
                search_status2=gr.Textbox(label='Search Status')
                search_results=gr.Dataframe(label='Similar Cases')
                search_gallery=gr.Gallery(label='Before & After Images',columns=2,height='auto')
                search_btn2.click(fn=search_cases,inputs=[search_plant],
                                  outputs=[search_status2,search_results,search_gallery])

app.launch()

/tmp/ipykernel_8708/2527668743.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as app:
/tmp/ipykernel_8708/2527668743.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2af08a4f62db100018.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
